# CareLoop Clinics: No-Show Guard
## Part 4: GenAI (turn call-centre notes into structured reasons, then recommend)

**Inputs:** `data/followup_notes.csv` (first 10 notes only, free-tier limit), `appointments_clean.csv` from Part 1.
In Colab, upload `data/followup_notes.csv` and `appointments_clean.csv` before running.
**Output:** `notes_analysed.csv`.

**API key safety:** the key is typed at run time (hidden prompt), never written in a cell or a file. Before downloading this notebook: *Edit → Clear all outputs* on the key cell, then *Edit → Find* the first characters of your key. There must be **0 matches**.

Every API call goes through one function that counts calls, so the total in Task 4.3(c) is what the code actually did.

In [1]:
!pip install -q google-genai pydantic

In [2]:
import json, time
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

# Use the Flash model name from your Session 22.1 notebook; if it has been retired,
# use the current Flash model listed in Google AI Studio.
MODEL_NAME = "gemini-3.5-flash-lite"

ALLOWED = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
           "illness_or_emergency", "work_or_schedule", "other"]

notes = pd.read_csv("data/followup_notes.csv")
print(notes.shape)
print(notes.head(10).to_string())

(30, 3)
  note_id appointment_id                                                                                                                      note_text
0    N001          A0872          Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
1    N002          A0897    Lives far from the clinic and could not find any transport that morning. Says the clinic is too far for a routine check-up.
2    N003          A0526  Patient felt the consultation fee was too high this month and decided to skip. Asked if there is any discount for follow-ups.
3    N004          A0926                                                                                                                            NaN
4    N005          A0534                                             Had high fever and body ache on the day, stayed in bed. Will visit once recovered.
5    N006          A0507                     Manager called an urgent meeting at

### API key (typed at run time, never saved)
`getpass` is `input()` that hides what you type, so the key never appears in the cell output.

In [3]:
from getpass import getpass
client = genai.Client(api_key=getpass("Paste your Gemini API key: "),
                      http_options=types.HttpOptions(timeout=180000))   # 60-second limit per call
print("Client ready.")

Paste your Gemini API key: ··········
Client ready.


## Task 4.1: One validated call
The JSON response schema has four fields. `root_cause` is a plain string, **not** an enum: the allowed values are in the prompt and are enforced afterwards by the check in Task 4.2.

In [5]:
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

SYSTEM_TEXT = ("You are a careful analyst for an outpatient clinic chain. You read short call-centre notes "
               "and return only the requested JSON. You never invent facts that are not in the note.")

def craft_prompt(note_text):
    """Prompt built from the CRAFT parts: Context, Role, Action, Tone, Format."""
    return f"""
CONTEXT: CareLoop Clinics runs eight outpatient clinics in India. After a patient misses an appointment,
the call centre phones them and writes a short note. Notes may mix English and Hindi (Hinglish).

ROLE: You are a patient-experience analyst who classifies why patients missed their appointment.

ACTION: Read the note below and decide the single main reason the patient missed the visit.
root_cause MUST be exactly one of these values:
- forgot_or_no_reminder  (forgot, did not get / did not see a reminder, mixed up the date)
- distance_or_transport  (too far, no transport, traffic, vehicle problem)
- cost_or_fee            (fee too high, could not afford, payment problem)
- illness_or_emergency   (patient or family member ill, hospitalised, emergency)
- work_or_schedule       (work, exams, travel or another commitment clashed)
- other                  (anything else, OR if the note is ambiguous or gives no clear reason)
If the reason is not clear from the note, answer "other". Do not guess.
confidence is a number from 0 to 1 showing how sure you are (use a low value for "other" due to ambiguity).
summary: what happened, in at most 25 words, in English.
recommended_action: one concrete next step for the clinic.

TONE: Neutral, factual and respectful to the patient.

FORMAT: Return only JSON with the keys root_cause, confidence, summary, recommended_action.

NOTE: \"\"\"{note_text}\"\"\"
""".strip()

API_CALLS = 0

from google.genai import errors

def call_gemini(prompt):
    """Send one prompt with the fixed settings; returns the raw JSON text.
    Counts every call, including up to 3 retries if Google's server is busy (5xx)."""
    global API_CALLS
    for attempt in range(1, 5):
        API_CALLS += 1
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_TEXT,
                    temperature=0,
                    max_output_tokens=1000,
                    response_mime_type="application/json",
                    response_schema=SCHEMA,
                    thinking_config=types.ThinkingConfig(thinking_level="minimal"),  # if your model rejects this, delete this line
                ),
            )
            return response.text
        except errors.ServerError as e:
            if attempt == 4:
                raise
            print(f"   server error {e.code}, retry {attempt} in {30 * attempt} s")
            time.sleep(30 * attempt)

def analyse_note(note_text):
    """Classify one note; returns (parsed dict, raw reply text)."""
    raw = call_gemini(craft_prompt(note_text))
    return json.loads(raw), raw

n001 = notes.loc[notes["note_id"] == "N001", "note_text"].iloc[0]
print("N001:", n001)
data, raw = analyse_note(n001)
print(json.dumps(data, indent=2))
print("Fields:", sorted(data.keys()))
print("API calls so far:", API_CALLS)

N001: Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
{
  "root_cause": "forgot_or_no_reminder",
  "confidence": 0.95,
  "summary": "Patient forgot the visit because she never received a reminder message, and wishes to reschedule for next week.",
  "recommended_action": "Reschedule the appointment for next week and verify the patient's contact number for future reminders."
}
Fields: ['confidence', 'recommended_action', 'root_cause', 'summary']
API calls so far: 1


## Task 4.2: Validate and process the first 10 notes
(a) Pydantic model plus allowed-list check. (b) Blank notes are skipped **without** an API call. (c) 13-second pause between calls. (d) Every reply is kept exactly as returned in `raw_response`, and invalid replies are never repaired.

In [6]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def check_allowed(insight):
    if insight.root_cause not in ALLOWED:
        raise ValueError(f"root_cause '{insight.root_cause}' is not in the allowed list")
    return insight

FIELDS = ["root_cause", "confidence", "summary", "recommended_action"]
results = []

for i, row in notes.head(10).iterrows():
    note_text = row["note_text"]
    rec = {"note_id": row["note_id"], "appointment_id": row["appointment_id"]}

    # input validation: NaN first (a float has no .strip()), then empty / only spaces
    if pd.isna(note_text) or str(note_text).strip() == "":
        rec.update(root_cause="other", confidence=0.0, summary="No note recorded",
                   recommended_action="Re-call patient", validation_status="skipped_blank", raw_response="")
        print(f"{row['note_id']}: blank note, skipped (no API call)")
        results.append(rec)
        continue

    if API_CALLS > 0:
        time.sleep(13)                      # free-tier rate limit
    raw = ""
    try:
        raw = call_gemini(craft_prompt(note_text))
        data = json.loads(raw)
        for f in FIELDS:                     # copy whatever the reply contained, as returned
            rec[f] = data.get(f, None)
        check_allowed(FollowupInsight(**data))
        rec["validation_status"] = "valid"
    except Exception as e:                   # one bad reply never stops the loop
        rec["validation_status"] = "invalid"
        print(f"   {row['note_id']} invalid: {type(e).__name__}: {str(e)[:200]}")
    rec["raw_response"] = raw
    results.append(rec)
    print(f"{row['note_id']}: {rec.get('root_cause')} ({rec['validation_status']})")

analysed = pd.DataFrame(results).reindex(columns=["note_id", "appointment_id", "root_cause", "confidence",
                                                  "summary", "recommended_action", "validation_status", "raw_response"])
analysed.to_csv("notes_analysed.csv", index=False)
print()
print("Saved notes_analysed.csv:", analysed.shape)
print(analysed[["note_id", "root_cause", "confidence", "validation_status"]].to_string(index=False))
print("API calls so far:", API_CALLS)

   server error 503, retry 1 in 30 s
N001: forgot_or_no_reminder (valid)
N002: distance_or_transport (valid)
N003: cost_or_fee (valid)
N004: blank note, skipped (no API call)
N005: illness_or_emergency (valid)
N006: work_or_schedule (valid)
   server error 503, retry 1 in 30 s
N007: forgot_or_no_reminder (valid)
N008: distance_or_transport (valid)
   server error 503, retry 1 in 30 s
   server error 503, retry 2 in 60 s
   server error 503, retry 3 in 90 s
N009: other (valid)
N010: work_or_schedule (valid)

Saved notes_analysed.csv: (10, 8)
note_id            root_cause  confidence validation_status
   N001 forgot_or_no_reminder        0.95             valid
   N002 distance_or_transport        0.95             valid
   N003           cost_or_fee        0.95             valid
   N004                 other        0.00     skipped_blank
   N005  illness_or_emergency        1.00             valid
   N006      work_or_schedule        0.95             valid
   N007 forgot_or_no_reminder    

## Task 4.3: Break it on purpose
### (a) Two deliberately bad replies

In [7]:
bad_replies = {
    "confidence = 1.7": {"root_cause": "cost_or_fee", "confidence": 1.7,
                         "summary": "Patient said the fee was too high.", "recommended_action": "Offer a fee plan."},
    "missing recommended_action": {"root_cause": "cost_or_fee", "confidence": 0.8,
                                   "summary": "Patient said the fee was too high."},
}
for name, bad in bad_replies.items():
    try:
        FollowupInsight(**bad)
        print(f"{name}: passed (unexpected)")
    except ValidationError as e:
        print(f"{name}: caught ValidationError")
        for err in e.errors():
            print(f"   field={err['loc']}, type={err['type']}, message={err['msg']}")

confidence = 1.7: caught ValidationError
   field=('confidence',), type=less_than_equal, message=Input should be less than or equal to 1
missing recommended_action: caught ValidationError
   field=('recommended_action',), type=missing, message=Field required


### (b) V1 vs V2 prompt on N007 (Hinglish) and N009 (ambiguous)
Same schema and settings for both (root_cause as a plain string, JSON mode, temperature 0), so the **only** difference is the prompt.
V2 is the CRAFT prompt already used in Task 4.2, with identical settings, so its results are reused from `notes_analysed.csv`. Only the two V1 calls are new.

In [8]:
V1_TEMPLATE = "Classify this note:\n{note}"

rows = []
for nid in ["N007", "N009"]:
    note_text = notes.loc[notes["note_id"] == nid, "note_text"].iloc[0]
    time.sleep(13)
    raw_v1 = call_gemini(V1_TEMPLATE.format(note=note_text))
    v1 = json.loads(raw_v1).get("root_cause")
    v2 = analysed.loc[analysed["note_id"] == nid, "root_cause"].iloc[0]
    rows.append({"note": nid, "prompt": "V1 (just 'Classify this note')", "root_cause": v1, "allowed?": v1 in ALLOWED})
    rows.append({"note": nid, "prompt": "V2 (CRAFT + allowed list + 'other')", "root_cause": v2, "allowed?": v2 in ALLOWED})

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

note                              prompt                                                                    root_cause  allowed?
N007      V1 (just 'Classify this note') Patient forgot the appointment because they did not receive the SMS reminder.     False
N007 V2 (CRAFT + allowed list + 'other')                                                         forgot_or_no_reminder      True
N009      V1 (just 'Classify this note')                                                           Patient unreachable     False
N009 V2 (CRAFT + allowed list + 'other')                                                                         other      True


**Which prompt to deploy: V2.** Check the `allowed?` column above. V1 gives the model no list, so it usually makes up its own label (for example "missed appointment" or "unreachable"). That fails the allowed-list check and can't be counted or cross-tabulated. V2 names the six allowed values and says to answer `other` rather than guess, so it returns an allowed value for both notes: the Hinglish note maps to `forgot_or_no_reminder`, and the ambiguous note (patient unreachable, no reason given) maps to `other`.

### (c) Total API calls used in Tasks 4.1–4.3

In [10]:
print("Total Gemini API calls:", API_CALLS)
print("  = 1 note in 4.1 + 9 notes in 4.2 (N004 blank, skipped) + 2 V1 calls in 4.3,")
print("    plus automatic retries after Gemini 503 'high demand' errors (each attempt counted)")
assert API_CALLS < 25

Total Gemini API calls: 17
  = 1 note in 4.1 + 9 notes in 4.2 (N004 blank, skipped) + 2 V1 calls in 4.3,
    plus automatic retries after Gemini 503 'high demand' errors (each attempt counted)


## Task 4.4(a): Combine the notes with the bookings

In [11]:
appts = pd.read_csv("appointments_clean.csv")
merged = analysed.merge(appts, on="appointment_id", how="left")
print("Merged rows:", len(merged))
print(merged[["note_id", "appointment_id", "root_cause", "reminder_sent", "lead_days", "clinic_id"]].to_string(index=False))
print()
print(pd.crosstab(merged["root_cause"], merged["reminder_sent"], margins=True))

Merged rows: 10
note_id appointment_id            root_cause reminder_sent  lead_days clinic_id
   N001          A0872 forgot_or_no_reminder            No         14       C05
   N002          A0897 distance_or_transport            No          5       C07
   N003          A0526           cost_or_fee           Yes         34       C04
   N004          A0926                 other            No         40       C04
   N005          A0534  illness_or_emergency           Yes         26       C04
   N006          A0507      work_or_schedule           Yes         28       C04
   N007          A0333 forgot_or_no_reminder            No          9       C08
   N008          A0222 distance_or_transport           Yes          4       C02
   N009          A0977                 other           Yes          4       C01
   N010          A0596      work_or_schedule            No         21       C04

reminder_sent          No  Yes  All
root_cause                         
cost_or_fee             0    1 